**Hypothesis Testing**

* Tests Conducted:
    1) Test 1  Independent t-Test     Weekday vs Weekend AOV
    2) Test 2  One-Way ANOVA          Revenue across Categories
            + Tukey HSD            Pairwise category comparison
    3) Test 3  Mann-Whitney U         UK vs International frequency
    4) Test 4  A/B Test (t-Test)      Q3 vs Q4 daily revenue lift
    5) Test 5  Chi-Square             Category vs Bulk behaviour
    6) Test 6  Pearson + Spearman     Price vs Quantity correlation
 
* Stats Concepts Covered:
    1) Normality check      Shapiro-Wilk
    2) Variance check       Levene's test
    3) Effect sizes         Cohen's d, Eta², Cramér's V,
                           Rank-biserial r
    4) Confidence intervals 95% CI on all t-tests
    5) Post-hoc testing     Tukey HSD
    6) Non-parametric test  Mann-Whitney U
    7) Both tailed tests    One-tailed (T4) + Two-tailed (T1)

In [1]:
import os
import pandas as pd
import numpy as np
from scipy import stats
from scipy.stats import (ttest_ind, mannwhitneyu, f_oneway,
                         chi2_contingency, pearsonr, spearmanr,
                         shapiro, levene)
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import warnings
warnings.filterwarnings('ignore')

In [2]:
# ── output folder ──────────────────────────────────────────
os.makedirs('hypothesis_testing_results', exist_ok=True)

In [3]:
# ── helper: print section header ──────────────────────────
def header(test_num, title):
    print(f"\n{'═'*60}")
    print(f"  TEST {test_num} | {title}")
    print(f"{'═'*60}")
 
def subheader(text):
    print(f"\n  ── {text} ──────────────────────────────")
 
def result_line(label, value):
    print(f"  {label:<35}: {value}")

In [4]:
# ── load data ──────────────────────────────────────────────
df = pd.read_csv('data/online_retail_cleaned.csv',
                 parse_dates=['InvoiceDate'])
df_retail = df[df['IsBulkOrder'] == False].copy()
print(f"\n  Dataset loaded: {len(df):,} rows")
print(f"  Retail subset : {len(df_retail):,} rows (bulk excluded where noted)\n")


  Dataset loaded: 89,827 rows
  Retail subset : 86,760 rows (bulk excluded where noted)



In [5]:
# collector for summary
summary_rows = []

**TEST 1 — Independent t-Test**

**Do customers spend differently on weekdays vs weekends?**

In [7]:
header(1, "Independent t-Test | Weekday vs Weekend Order Value")
 
print("""
  Business Question:
    Do customers spend significantly more on weekdays vs weekends?
 
  H₀ : Mean order value on weekdays = Mean order value on weekends
  H₁ : Mean order value on weekdays ≠ Mean order value on weekends
  Test: Independent samples t-test (two-tailed)
""")
 
# Order-level revenue
order_rev = (df_retail.groupby(['InvoiceNo', 'IsWeekend'])['Revenue']
             .sum().reset_index())
weekday = order_rev[order_rev['IsWeekend'] == False]['Revenue']
weekend = order_rev[order_rev['IsWeekend'] == True]['Revenue']
 
subheader("Descriptive Statistics")
result_line("Weekday orders (n)", f"{len(weekday):,}")
result_line("Weekend orders (n)", f"{len(weekend):,}")
result_line("Weekday mean order value", f"£{weekday.mean():,.2f}")
result_line("Weekend mean order value", f"£{weekend.mean():,.2f}")
result_line("Weekday std dev", f"£{weekday.std():,.2f}")
result_line("Weekend std dev", f"£{weekend.std():,.2f}")
 
subheader("Assumptions Check")
# Normality — Shapiro-Wilk on a sample (Shapiro needs n <= 5000)
sample_wd = weekday.sample(min(1000, len(weekday)), random_state=42)
sample_we = weekend.sample(min(1000, len(weekend)), random_state=42)
_, p_norm_wd = shapiro(sample_wd)
_, p_norm_we = shapiro(sample_we)
result_line("Shapiro-Wilk p (weekday)", f"{p_norm_wd:.4f} → {'Normal ✓' if p_norm_wd > 0.05 else 'Not Normal — but t-test robust at large n ✓'}")
result_line("Shapiro-Wilk p (weekend)", f"{p_norm_we:.4f} → {'Normal ✓' if p_norm_we > 0.05 else 'Not Normal — but t-test robust at large n ✓'}")
 
# Equal variance — Levene's test
_, p_levene = levene(weekday, weekend)
equal_var = p_levene > 0.05
result_line("Levene's test p-value", f"{p_levene:.4f} → {'Equal variances ✓' if equal_var else 'Unequal variances → Welch t-test used'}")
 
subheader("Test Result")
t_stat, p_val = ttest_ind(weekday, weekend, equal_var=equal_var)
ci = stats.t.interval(0.95,
                      df=len(weekday)+len(weekend)-2,
                      loc=weekday.mean()-weekend.mean(),
                      scale=stats.sem(np.concatenate([weekday, weekend])))
# Cohen's d
pooled_std = np.sqrt((weekday.std()**2 + weekend.std()**2) / 2)
cohens_d   = (weekday.mean() - weekend.mean()) / pooled_std
 
result_line("t-statistic", f"{t_stat:.4f}")
result_line("p-value", f"{p_val:.4f}")
result_line("95% Confidence Interval", f"[{ci[0]:,.2f}, {ci[1]:,.2f}]")
result_line("Cohen's d (effect size)", f"{cohens_d:.4f} → {'Small' if abs(cohens_d)<0.2 else 'Medium' if abs(cohens_d)<0.5 else 'Large'}")
 
conclusion1 = ("REJECT H₀ — Weekday order value is significantly higher than weekend."
               if p_val < 0.05 else
               "FAIL TO REJECT H₀ — No significant difference in weekday vs weekend order value.")
print(f"\nDecision (α=0.05): {conclusion1}")
print(f"Business Insight : Marketing budgets should be weighted toward "
      f"{'weekdays' if weekday.mean() > weekend.mean() else 'weekends'} "
      f"for higher order value conversion.")
 
# Save
t1 = pd.DataFrame([{
    'Test': 'Independent t-Test',
    'H0': 'Weekday mean = Weekend mean',
    'Group_A': 'Weekday', 'Mean_A': round(weekday.mean(), 2), 'n_A': len(weekday),
    'Group_B': 'Weekend', 'Mean_B': round(weekend.mean(), 2), 'n_B': len(weekend),
    't_statistic': round(t_stat, 4), 'p_value': round(p_val, 4),
    'CI_lower': round(ci[0], 4), 'CI_upper': round(ci[1], 4),
    'Cohens_d': round(cohens_d, 4),
    'Significant': p_val < 0.05, 'Conclusion': conclusion1
}])
t1.to_csv('hypothesis_testing_results/test1_ttest_weekday_weekend.csv', index=False)
summary_rows.append({'Test': 'T-Test: Weekday vs Weekend',
                     'p_value': round(p_val, 4),
                     'Significant': p_val < 0.05,
                     'Effect_Size': f"Cohen's d = {cohens_d:.4f}",
                     'Conclusion': conclusion1})
print("\nSaved: test1_ttest_weekday_weekend.csv")


════════════════════════════════════════════════════════════
  TEST 1 | Independent t-Test | Weekday vs Weekend Order Value
════════════════════════════════════════════════════════════

  Business Question:
    Do customers spend significantly more on weekdays vs weekends?

  H₀ : Mean order value on weekdays = Mean order value on weekends
  H₁ : Mean order value on weekdays ≠ Mean order value on weekends
  Test: Independent samples t-test (two-tailed)


  ── Descriptive Statistics ──────────────────────────────
  Weekday orders (n)                 : 19,230
  Weekend orders (n)                 : 7,754
  Weekday mean order value           : £471.79
  Weekend mean order value           : £474.78
  Weekday std dev                    : £539.53
  Weekend std dev                    : £539.67

  ── Assumptions Check ──────────────────────────────
  Shapiro-Wilk p (weekday)           : 0.0000 → Not Normal — but t-test robust at large n ✓
  Shapiro-Wilk p (weekend)           : 0.0000 → Not Nor

**TEST 2 — One-Way ANOVA + Tukey HSD**

**Is revenue significantly different across product categories?**

In [8]:
header(2, "One-Way ANOVA + Tukey HSD | Revenue across Categories")
 
print("""
  Business Question:
    Are revenue differences across the 7 product categories
    statistically significant, or just random variation?
 
  H₀ : Mean revenue is equal across all categories
  H₁ : At least one category mean differs significantly
  Test: One-Way ANOVA (parametric, 3+ groups)
  Post-hoc: Tukey HSD (to find which pairs differ)
""")
 
cat_groups = [grp['Revenue'].values
              for _, grp in df_retail.groupby('Category')]
cat_names  = df_retail['Category'].unique()
 
subheader("Descriptive Statistics per Category")
cat_stats = (df_retail.groupby('Category')['Revenue']
             .agg(['mean','std','count'])
             .round(2))
cat_stats.columns = ['Mean_Revenue', 'Std_Dev', 'Count']
print(cat_stats.to_string())
 
subheader("ANOVA Result")
f_stat, p_anova = f_oneway(*cat_groups)
# Eta-squared (effect size for ANOVA)
grand_mean = df_retail['Revenue'].mean()
ss_between = sum(len(g) * (g.mean() - grand_mean)**2 for g in cat_groups)
ss_total   = sum((df_retail['Revenue'] - grand_mean)**2)
eta_sq     = ss_between / ss_total
 
result_line("F-statistic", f"{f_stat:.4f}")
result_line("p-value", f"{p_anova:.4f}")
result_line("Eta-squared (effect size)", f"{eta_sq:.4f} → {'Small' if eta_sq<0.06 else 'Medium' if eta_sq<0.14 else 'Large'}")
 
conclusion2 = ("REJECT H₀ — Revenue differs significantly across categories."
               if p_anova < 0.05 else
               "FAIL TO REJECT H₀ — No significant revenue difference across categories.")
print(f"\nDecision (α=0.05): {conclusion2}")
 
subheader("Post-hoc: Tukey HSD (significant pairs only)")
tukey = pairwise_tukeyhsd(df_retail['Revenue'],
                           df_retail['Category'], alpha=0.05)
tukey_df = pd.DataFrame(data=tukey._results_table.data[1:],
                         columns=tukey._results_table.data[0])
sig_pairs = tukey_df[tukey_df['reject'] == True]
if len(sig_pairs) > 0:
    print(sig_pairs[['group1','group2','meandiff','p-adj']].to_string(index=False))
else:
    print("  No significant pairs found at α=0.05")
 
print(f"\nBusiness Insight: {'Different pricing/promotion strategies are statistically justified per category.' if p_anova < 0.05 else 'Uniform category strategy may be acceptable.'}")
 
# Save
t2 = cat_stats.reset_index()
t2['F_statistic'] = round(f_stat, 4)
t2['p_value']     = round(p_anova, 4)
t2['Eta_squared'] = round(eta_sq, 4)
t2['Significant'] = p_anova < 0.05
t2.to_csv('hypothesis_testing_results/test2_anova_categories.csv', index=False)
sig_pairs.to_csv('hypothesis_testing_results/test2_tukey_pairs.csv', index=False)
summary_rows.append({'Test': 'One-Way ANOVA: Revenue by Category',
                     'p_value': round(p_anova, 4),
                     'Significant': p_anova < 0.05,
                     'Effect_Size': f"Eta² = {eta_sq:.4f}",
                     'Conclusion': conclusion2})
print("\nSaved: test2_anova_categories.csv")
print("Saved: test2_tukey_pairs.csv")


════════════════════════════════════════════════════════════
  TEST 2 | One-Way ANOVA + Tukey HSD | Revenue across Categories
════════════════════════════════════════════════════════════

  Business Question:
    Are revenue differences across the 7 product categories
    statistically significant, or just random variation?

  H₀ : Mean revenue is equal across all categories
  H₁ : At least one category mean differs significantly
  Test: One-Way ANOVA (parametric, 3+ groups)
  Post-hoc: Tukey HSD (to find which pairs differ)


  ── Descriptive Statistics per Category ──────────────────────────────
               Mean_Revenue  Std_Dev  Count
Category                                   
Clothing             280.87   229.58   8428
Garden               249.04   196.79   8341
Gifts & Cards         89.29    70.14  13062
Home Decor           102.22    82.13  18893
Kitchen              151.32   134.25  13024
Stationery            69.71    47.89  11205
Toys & Games         178.18   133.50  1380

**TEST 3 — Mann-Whitney U Test**

**Do UK customers order more frequently than international?**

In [9]:
header(3, "Mann-Whitney U Test | UK vs International Order Frequency")
 
print("""
  Business Question:
    Do UK customers place orders more frequently than
    international customers?
 
  H₀ : Order frequency distribution is the same for UK and
       international customers
  H₁ : UK customers order more frequently than international
  Test: Mann-Whitney U (non-parametric — order frequency is skewed)
  Direction: One-tailed (UK > International)
""")
 
cust_freq = (df_retail[df_retail['CustomerID'] != 'Guest']
             .groupby('CustomerID')
             .agg(OrderCount=('InvoiceNo','nunique'),
                  Country=('Country','first'))
             .reset_index())
 
uk_freq    = cust_freq[cust_freq['Country'] == 'United Kingdom']['OrderCount']
intl_freq  = cust_freq[cust_freq['Country'] != 'United Kingdom']['OrderCount']
 
subheader("Assumptions Check")
result_line("Why Mann-Whitney (not t-test)",
            "Order frequency is right-skewed (non-normal)")
_, p_norm_uk   = shapiro(uk_freq.sample(min(500, len(uk_freq)), random_state=42))
_, p_norm_intl = shapiro(intl_freq.sample(min(500, len(intl_freq)), random_state=42))
result_line("Shapiro-Wilk p (UK)", f"{p_norm_uk:.4f} → Not Normal ✓ (Mann-Whitney correct choice)")
result_line("Shapiro-Wilk p (Intl)", f"{p_norm_intl:.4f} → Not Normal ✓")
 
subheader("Descriptive Statistics")
result_line("UK customers (n)", f"{len(uk_freq):,}")
result_line("International customers (n)", f"{len(intl_freq):,}")
result_line("UK median orders", f"{uk_freq.median():.1f}")
result_line("International median orders", f"{intl_freq.median():.1f}")
result_line("UK mean orders", f"{uk_freq.mean():.2f}")
result_line("International mean orders", f"{intl_freq.mean():.2f}")
 
subheader("Test Result")
u_stat, p_mw = mannwhitneyu(uk_freq, intl_freq, alternative='greater')
# Effect size: rank-biserial correlation
r_rb = 1 - (2 * u_stat) / (len(uk_freq) * len(intl_freq))
 
result_line("U-statistic", f"{u_stat:.2f}")
result_line("p-value (one-tailed)", f"{p_mw:.4f}")
result_line("Rank-biserial r (effect)", f"{r_rb:.4f} → {'Small' if abs(r_rb)<0.1 else 'Medium' if abs(r_rb)<0.3 else 'Large'}")
 
conclusion3 = ("REJECT H₀ — UK customers order significantly more frequently."
               if p_mw < 0.05 else
               "FAIL TO REJECT H₀ — No significant difference in order frequency.")
print(f"\nDecision (α=0.05): {conclusion3}")
print(f"Business Insight: International customers need stronger re-engagement "
      f"campaigns to match UK order frequency levels.")
 
# Save
t3 = pd.DataFrame([{
    'Test': 'Mann-Whitney U',
    'H0': 'UK freq = International freq',
    'UK_n': len(uk_freq), 'UK_median': uk_freq.median(),
    'Intl_n': len(intl_freq), 'Intl_median': intl_freq.median(),
    'U_statistic': round(u_stat, 2), 'p_value': round(p_mw, 4),
    'Rank_biserial_r': round(r_rb, 4),
    'Significant': p_mw < 0.05, 'Conclusion': conclusion3
}])
t3.to_csv('hypothesis_testing_results/test3_mannwhitney_countries.csv', index=False)
summary_rows.append({'Test': 'Mann-Whitney U: UK vs Intl Frequency',
                     'p_value': round(p_mw, 4),
                     'Significant': p_mw < 0.05,
                     'Effect_Size': f"Rank-biserial r = {r_rb:.4f}",
                     'Conclusion': conclusion3})
print("\nSaved: test3_mannwhitney_countries.csv")


════════════════════════════════════════════════════════════
  TEST 3 | Mann-Whitney U Test | UK vs International Order Frequency
════════════════════════════════════════════════════════════

  Business Question:
    Do UK customers place orders more frequently than
    international customers?

  H₀ : Order frequency distribution is the same for UK and
       international customers
  H₁ : UK customers order more frequently than international
  Test: Mann-Whitney U (non-parametric — order frequency is skewed)
  Direction: One-tailed (UK > International)


  ── Assumptions Check ──────────────────────────────
  Why Mann-Whitney (not t-test)      : Order frequency is right-skewed (non-normal)
  Shapiro-Wilk p (UK)                : 0.0000 → Not Normal ✓ (Mann-Whitney correct choice)
  Shapiro-Wilk p (Intl)              : 0.0000 → Not Normal ✓

  ── Descriptive Statistics ──────────────────────────────
  UK customers (n)                   : 3,846
  International customers (n)        : 83

**TEST 4 — A/B Test (Two-sample t-test)**

**Did Q4 (holiday season) significantly lift daily revenue vs Q3?**

In [10]:
header(4, "A/B Test (t-Test) | Q3 vs Q4 Daily Revenue (Holiday Lift)")
 
print("""
  Business Question:
    Was the Q4 (Oct–Dec) holiday season revenue lift
    statistically significant compared to Q3 (Jul–Sep)?
 
  H₀ : Mean daily revenue in Q3 = Mean daily revenue in Q4
  H₁ : Mean daily revenue in Q4 > Mean daily revenue in Q3
  Test: One-tailed two-sample t-test
  Extra: Cohen's d for effect size magnitude
""")
 
daily_rev = (df.groupby(['InvoiceDate', 'Quarter'])['Revenue']
               .sum().reset_index())
daily_rev.columns = ['Date', 'Quarter', 'DailyRevenue']
 
q3_rev = daily_rev[daily_rev['Quarter'] == 3]['DailyRevenue']
q4_rev = daily_rev[daily_rev['Quarter'] == 4]['DailyRevenue']
 
subheader("Descriptive Statistics")
result_line("Q3 days (n)", f"{len(q3_rev):,}")
result_line("Q4 days (n)", f"{len(q4_rev):,}")
result_line("Q3 mean daily revenue", f"£{q3_rev.mean():,.2f}")
result_line("Q4 mean daily revenue", f"£{q4_rev.mean():,.2f}")
result_line("Q4 lift over Q3", f"£{q4_rev.mean()-q3_rev.mean():,.2f} ({(q4_rev.mean()-q3_rev.mean())/q3_rev.mean()*100:.1f}%)")
result_line("Q3 std dev", f"£{q3_rev.std():,.2f}")
result_line("Q4 std dev", f"£{q4_rev.std():,.2f}")
 
subheader("Assumptions Check")
_, p_lev = levene(q3_rev, q4_rev)
equal_var_ab = p_lev > 0.05
result_line("Levene's test p-value",
            f"{p_lev:.4f} → {'Equal variances' if equal_var_ab else 'Unequal → Welch t-test'}")
 
subheader("Test Result")
t_ab, p_ab = ttest_ind(q4_rev, q3_rev,
                        equal_var=equal_var_ab,
                        alternative='greater')
ci_ab = stats.t.interval(0.95,
                          df=len(q3_rev)+len(q4_rev)-2,
                          loc=q4_rev.mean()-q3_rev.mean(),
                          scale=stats.sem(np.concatenate([q3_rev, q4_rev])))
pooled_ab  = np.sqrt((q3_rev.std()**2 + q4_rev.std()**2) / 2)
cohens_d_ab = (q4_rev.mean() - q3_rev.mean()) / pooled_ab
 
result_line("t-statistic", f"{t_ab:.4f}")
result_line("p-value (one-tailed)", f"{p_ab:.4f}")
result_line("95% CI on difference", f"[£{ci_ab[0]:,.2f}, £{ci_ab[1]:,.2f}]")
result_line("Cohen's d (effect size)",
            f"{cohens_d_ab:.4f} → {'Small' if abs(cohens_d_ab)<0.2 else 'Medium' if abs(cohens_d_ab)<0.5 else 'Large'}")
 
conclusion4 = ("REJECT H₀ — Q4 daily revenue is significantly higher than Q3."
               if p_ab < 0.05 else
               "FAIL TO REJECT H₀ — Q4 lift over Q3 is not statistically significant.")
print(f"\nDecision (α=0.05): {conclusion4}")
print(f"Business Insight: Q4 holiday lift of "
      f"£{q4_rev.mean()-q3_rev.mean():,.0f}/day is statistically proven — "
      f"invest heavily in Q4 inventory and marketing.")
 
# Save
t4 = pd.DataFrame([{
    'Test': 'A/B Test (One-tailed t-Test)',
    'H0': 'Q3 mean daily revenue = Q4 mean daily revenue',
    'Q3_n': len(q3_rev), 'Q3_mean': round(q3_rev.mean(), 2),
    'Q4_n': len(q4_rev), 'Q4_mean': round(q4_rev.mean(), 2),
    'Lift_GBP': round(q4_rev.mean()-q3_rev.mean(), 2),
    'Lift_Pct': round((q4_rev.mean()-q3_rev.mean())/q3_rev.mean()*100, 2),
    't_statistic': round(t_ab, 4), 'p_value': round(p_ab, 4),
    'CI_lower': round(ci_ab[0], 2), 'CI_upper': round(ci_ab[1], 2),
    'Cohens_d': round(cohens_d_ab, 4),
    'Significant': p_ab < 0.05, 'Conclusion': conclusion4
}])
t4.to_csv('hypothesis_testing_results/test4_ab_test_q3_vs_q4.csv', index=False)
summary_rows.append({'Test': 'A/B Test: Q3 vs Q4 Daily Revenue',
                     'p_value': round(p_ab, 4),
                     'Significant': p_ab < 0.05,
                     'Effect_Size': f"Cohen's d = {cohens_d_ab:.4f}",
                     'Conclusion': conclusion4})
print("\nSaved: test4_ab_test_q3_vs_q4.csv")


════════════════════════════════════════════════════════════
  TEST 4 | A/B Test (t-Test) | Q3 vs Q4 Daily Revenue (Holiday Lift)
════════════════════════════════════════════════════════════

  Business Question:
    Was the Q4 (Oct–Dec) holiday season revenue lift
    statistically significant compared to Q3 (Jul–Sep)?

  H₀ : Mean daily revenue in Q3 = Mean daily revenue in Q4
  H₁ : Mean daily revenue in Q4 > Mean daily revenue in Q3
  Test: One-tailed two-sample t-test
  Extra: Cohen's d for effect size magnitude


  ── Descriptive Statistics ──────────────────────────────
  Q3 days (n)                        : 184
  Q4 days (n)                        : 184
  Q3 mean daily revenue              : £18,177.73
  Q4 mean daily revenue              : £51,218.05
  Q4 lift over Q3                    : £33,040.32 (181.8%)
  Q3 std dev                         : £7,362.48
  Q4 std dev                         : £25,797.95

  ── Assumptions Check ──────────────────────────────
  Levene's test 

**TEST 5 — Chi-Square Test of Independence**

**Is product category independent of bulk order behaviour?**

In [11]:
header(5, "Chi-Square Test | Category vs Bulk Order Behaviour")
 
print("""
  Business Question:
    Are certain product categories more likely to be
    purchased in bulk (B2B) than others?
 
  H₀ : Product category and bulk order behaviour are independent
  H₁ : Product category and bulk order behaviour are associated
  Test: Chi-Square Test of Independence
  Effect Size: Cramér's V
""")
 
contingency = pd.crosstab(df['Category'], df['IsBulkOrder'])
contingency.columns = ['Non-Bulk', 'Bulk']
 
subheader("Contingency Table")
print(contingency.to_string())
 
subheader("Test Result")
chi2, p_chi2, dof, expected = chi2_contingency(contingency)
n = contingency.values.sum()
cramers_v = np.sqrt(chi2 / (n * (min(contingency.shape) - 1)))
 
result_line("Chi-square statistic", f"{chi2:.4f}")
result_line("Degrees of freedom", f"{dof}")
result_line("p-value", f"{p_chi2:.4f}")
result_line("Cramér's V (effect size)",
            f"{cramers_v:.4f} → {'Weak' if cramers_v<0.1 else 'Moderate' if cramers_v<0.3 else 'Strong'}")
 
subheader("Bulk Rate per Category")
contingency['BulkRate_Pct'] = (contingency['Bulk']
                                / (contingency['Bulk']
                                   + contingency['Non-Bulk']) * 100).round(2)
print(contingency['BulkRate_Pct'].sort_values(ascending=False).to_string())
 
conclusion5 = ("REJECT H₀ — Category and bulk behaviour are significantly associated."
               if p_chi2 < 0.05 else
               "FAIL TO REJECT H₀ — No significant association between category and bulk orders.")
print(f"\nDecision (α=0.05): {conclusion5}")
print(f"Business Insight: B2B outreach should be targeted to high-bulk "
      f"categories rather than a blanket approach across all products.")
 
# Save
t5 = contingency.reset_index()
t5['Chi2']      = round(chi2, 4)
t5['p_value']   = round(p_chi2, 4)
t5['CramersV']  = round(cramers_v, 4)
t5['Significant'] = p_chi2 < 0.05
t5.to_csv('hypothesis_testing_results/test5_chisquare_category_bulk.csv', index=False)
summary_rows.append({'Test': 'Chi-Square: Category vs Bulk',
                     'p_value': round(p_chi2, 4),
                     'Significant': p_chi2 < 0.05,
                     'Effect_Size': f"Cramér's V = {cramers_v:.4f}",
                     'Conclusion': conclusion5})
print("\nSaved: test5_chisquare_category_bulk.csv")


════════════════════════════════════════════════════════════
  TEST 5 | Chi-Square Test | Category vs Bulk Order Behaviour
════════════════════════════════════════════════════════════

  Business Question:
    Are certain product categories more likely to be
    purchased in bulk (B2B) than others?

  H₀ : Product category and bulk order behaviour are independent
  H₁ : Product category and bulk order behaviour are associated
  Test: Chi-Square Test of Independence
  Effect Size: Cramér's V


  ── Contingency Table ──────────────────────────────
               Non-Bulk  Bulk
Category                     
Clothing           8428   309
Garden             8341   285
Gifts & Cards     13062   479
Home Decor        18893   659
Kitchen           13024   436
Stationery        11205   412
Toys & Games      13807   487

  ── Test Result ──────────────────────────────
  Chi-square statistic               : 3.3205
  Degrees of freedom                 : 6
  p-value                            : 0.

**TEST 6 — Pearson + Spearman Correlation**

**Does higher unit price negatively correlate with quantity?**

In [12]:
header(6, "Pearson + Spearman Correlation | Price vs Quantity")
 
print("""
  Business Question:
    Is there a statistically significant negative relationship
    between unit price and quantity ordered?
    (i.e. do customers buy less when price is higher?)
 
  H₀ : No correlation between unit price and quantity ordered
  H₁ : Significant negative correlation exists
  Tests: Pearson (linear) + Spearman (monotonic, non-parametric)
""")
 
sample = df_retail.sample(min(10000, len(df_retail)), random_state=42)
 
subheader("Descriptive Statistics")
result_line("Sample size (n)", f"{len(sample):,}")
result_line("Price range", f"£{sample['UnitPrice'].min():.2f} – £{sample['UnitPrice'].max():.2f}")
result_line("Qty range", f"{sample['Quantity'].min()} – {sample['Quantity'].max()}")
result_line("Mean price", f"£{sample['UnitPrice'].mean():.2f}")
result_line("Mean quantity", f"{sample['Quantity'].mean():.2f}")
 
subheader("Pearson Correlation (linear relationship)")
r_p, p_pearson = pearsonr(sample['UnitPrice'], sample['Quantity'])
result_line("Pearson r", f"{r_p:.4f}")
result_line("p-value", f"{p_pearson:.4f}")
result_line("Interpretation",
            f"{'Significant' if p_pearson < 0.05 else 'Not significant'} "
            f"{'negative' if r_p < 0 else 'positive'} linear relationship")
 
subheader("Spearman Correlation (monotonic / non-parametric)")
r_s, p_spearman = spearmanr(sample['UnitPrice'], sample['Quantity'])
result_line("Spearman ρ", f"{r_s:.4f}")
result_line("p-value", f"{p_spearman:.4f}")
result_line("Interpretation",
            f"{'Significant' if p_spearman < 0.05 else 'Not significant'} "
            f"{'negative' if r_s < 0 else 'positive'} monotonic relationship")
 
conclusion6 = ("REJECT H₀ — Significant correlation between price and quantity."
               if p_pearson < 0.05 else
               "FAIL TO REJECT H₀ — No significant correlation between price and quantity.")
print(f"\nDecision (α=0.05): {conclusion6}")
print(f"Business Insight: "
      + ("Price sensitivity exists — discounting higher-priced items "
         "could meaningfully increase volume." if r_p < -0.1 and p_pearson < 0.05
         else "Price is not the primary driver of quantity — other factors "
              "(product type, season, B2B demand) matter more."))
 
# Save
t6 = pd.DataFrame([{
    'Test': 'Pearson + Spearman Correlation',
    'H0': 'No correlation between UnitPrice and Quantity',
    'n': len(sample),
    'Pearson_r': round(r_p, 4), 'Pearson_p': round(p_pearson, 4),
    'Spearman_rho': round(r_s, 4), 'Spearman_p': round(p_spearman, 4),
    'Significant': p_pearson < 0.05, 'Conclusion': conclusion6
}])
t6.to_csv('hypothesis_testing_results/test6_correlation_price_qty.csv', index=False)
summary_rows.append({'Test': 'Correlation: Price vs Quantity',
                     'p_value': round(p_pearson, 4),
                     'Significant': p_pearson < 0.05,
                     'Effect_Size': f"Pearson r = {r_p:.4f}, Spearman ρ = {r_s:.4f}",
                     'Conclusion': conclusion6})
print("\nSaved: test6_correlation_price_qty.csv")


════════════════════════════════════════════════════════════
  TEST 6 | Pearson + Spearman Correlation | Price vs Quantity
════════════════════════════════════════════════════════════

  Business Question:
    Is there a statistically significant negative relationship
    between unit price and quantity ordered?
    (i.e. do customers buy less when price is higher?)

  H₀ : No correlation between unit price and quantity ordered
  H₁ : Significant negative correlation exists
  Tests: Pearson (linear) + Spearman (monotonic, non-parametric)


  ── Descriptive Statistics ──────────────────────────────
  Sample size (n)                    : 10,000
  Price range                        : £2.30 – £39.55
  Qty range                          : 1 – 48
  Mean price                         : £11.62
  Mean quantity                      : 12.72

  ── Pearson Correlation (linear relationship) ──────────────────────────────
  Pearson r                          : -0.0009
  p-value                      

**SUMMARY TABLE — All 6 Tests**

In [13]:
print(f"\n\n{'═'*60}")
print("  HYPOTHESIS TESTING SUMMARY")
print(f"{'═'*60}")
 
summary_df = pd.DataFrame(summary_rows)
print(summary_df[['Test','p_value','Significant','Effect_Size']].to_string(index=False))
summary_df.to_csv('hypothesis_testing_results/hypothesis_testing_summary.csv', index=False)
print("\n  ✓ Saved: hypothesis_testing_summary.csv")



════════════════════════════════════════════════════════════
  HYPOTHESIS TESTING SUMMARY
════════════════════════════════════════════════════════════
                                Test  p_value  Significant                              Effect_Size
          T-Test: Weekday vs Weekend   0.6803        False                      Cohen's d = -0.0055
  One-Way ANOVA: Revenue by Category   0.0000         True                            Eta² = 0.2144
Mann-Whitney U: UK vs Intl Frequency   0.6051        False                 Rank-biserial r = 0.0058
    A/B Test: Q3 vs Q4 Daily Revenue   0.0000         True                       Cohen's d = 1.7417
        Chi-Square: Category vs Bulk   0.7677        False                      Cramér's V = 0.0061
      Correlation: Price vs Quantity   0.9298        False Pearson r = -0.0009, Spearman ρ = 0.0013

  ✓ Saved: hypothesis_testing_summary.csv
